# Galaxy Morphology with a Transformer/CNN Ensemble

物理本科机器学习课程项目，原始报告完成于2026年4月。

This notebook preserves the original training settings for ViT-B/16, Swin-T and EfficientNet-V2-S, followed by weighted five-view test-time augmentation and error analysis.

Run from top to bottom after installing the dependencies and obtaining the Galaxy Zoo competition data. Training follows the original CUDA/AMP workflow; CPU execution has not been validated. The recorded original hardware was an RTX 5070 Laptop GPU with 8 GB VRAM.

The eleven heads are **independent linear regression heads**, not hard-constrained probabilities or an explicitly enforced decision tree. Reported metrics refer only to the original local validation split. See `docs/PROJECT_NOTES.md` for provenance and limitations.

## 1. Data loading and augmentation

In [ ]:
import os
from pathlib import Path
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.std import tqdm 
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" 计算设备: {device}")

class GalaxyDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(int(self.df.iloc[idx, 0]))
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as exc:
            raise RuntimeError(f'Could not read image: {img_path}') from exc 
        labels = self.df.iloc[idx, 1:].values.astype('float32')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(labels)

train_transforms = transforms.Compose([
    transforms.RandomRotation(180),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def collate_fn_ignore_none(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = Path(os.environ.get('GALAXY_DATA_DIR', str(REPO_ROOT / 'data'))).expanduser().resolve()
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / 'training_solutions_rev1.csv'
IMG_DIR = DATA_DIR / 'images_training_rev1'
if not CSV_PATH.is_file() or not IMG_DIR.is_dir():
    raise FileNotFoundError('Place the training CSV and image folder in data/, or set GALAXY_DATA_DIR. See README.md.')

full_df = pd.read_csv(CSV_PATH)
train_df, val_df = train_test_split(full_df, test_size=0.2, random_state=42)

train_set = GalaxyDataset(train_df, IMG_DIR, transform=train_transforms)
val_set = GalaxyDataset(val_df, IMG_DIR, transform=val_transforms)

safe_train_loader = DataLoader(train_set, batch_size=8, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_fn_ignore_none)
val_loader = DataLoader(val_set, batch_size=8, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collate_fn_ignore_none)

print(" 区块一完成：数据加载和增强流程已就绪。")

## 2. Multi-task regression heads and training loop

The 37 continuous outputs are grouped into eleven task heads. The model does not impose simplex or conditional-probability constraints.

In [ ]:
class GZ2DecisionTreeModel(nn.Module):
    """Pretrained backbone with 11 independent linear heads and 37 total outputs."""
    def __init__(self, backbone, in_features):
        super().__init__()
        self.backbone = backbone
        
        self.task1 = nn.Linear(in_features, 3)  # Smooth, Features/Disk, Artifact
        self.task2 = nn.Linear(in_features, 2)  # Edge-on Yes/No
        self.task3 = nn.Linear(in_features, 2)  # Bar Yes/No
        self.task4 = nn.Linear(in_features, 2)  # Spiral Arms Yes/No
        self.task5 = nn.Linear(in_features, 4)  # Bulge Size (4 levels)
        self.task6 = nn.Linear(in_features, 2)  # Odd feature Yes/No
        self.task7 = nn.Linear(in_features, 3)  # Roundness (3 levels)
        self.task8 = nn.Linear(in_features, 7)  # Odd feature detail (7 types)
        self.task9 = nn.Linear(in_features, 3)  # Bulge Shape (3 types)
        self.task10 = nn.Linear(in_features, 3) # Arm Winding (3 levels)
        self.task11 = nn.Linear(in_features, 6) # Arm Count (6 levels)

    def forward(self, x):
        features = self.backbone(x)
        
        out = torch.cat([
            self.task1(features), self.task2(features), self.task3(features),
            self.task4(features), self.task5(features), self.task6(features),
            self.task7(features), self.task8(features), self.task9(features),
            self.task10(features), self.task11(features)
        ], dim=1)
        return out

def train_model_engine(model, model_name, epochs=15):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=8e-5, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')
    accumulation_steps = 4
    best_loss = float('inf')
    save_path = str(CHECKPOINT_DIR / f'galaxy_{model_name}_tree.pth')
    
    print(f"\n 开始训练 {model_name} (多任务回归) | 目标: {epochs} 轮")
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()
        
        pbar = tqdm(safe_train_loader, desc=f"{model_name} Epoch {epoch+1}/{epochs}", leave=True)
        for i, (images, labels) in enumerate(pbar):
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels) / accumulation_steps
            
            scaler.scale(loss).backward()
            
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(safe_train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            real_loss = loss.item() * accumulation_steps
            running_loss += real_loss * images.size(0)
            pbar.set_postfix({'MSE': f"{real_loss:.4f}"})
            
        scheduler.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            vbar = tqdm(val_loader, desc="Validating", leave=False)
            for images, labels in vbar:
                images, labels = images.to(device), labels.to(device)
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                
        epoch_val = val_loss / len(val_loader.dataset)
        print(f" {model_name} Epoch {epoch+1} | Val MSE: {epoch_val:.6f} | 耗时: {time.time()-start_time:.1f}s")
        
        if epoch_val < best_loss:
            best_loss = epoch_val
            torch.save(model.state_dict(), save_path)
            print(f"    [{model_name}] 模型权重已更新！RMSE: {np.sqrt(best_loss):.5f}")

print(" 区块二完成：多任务回归模型与训练引擎已定义。")

## 3. Train ViT-B/16

In [ ]:
torch.cuda.empty_cache()
gc.collect()

print(" 正在构建 ViT 多任务回归模型...")
vit_base = models.vit_b_16(weights='IMAGENET1K_V1')
in_ftrs_vit = vit_base.heads.head.in_features
vit_base.heads.head = nn.Identity() 

model_vit = GZ2DecisionTreeModel(vit_base, in_ftrs_vit).to(device)

train_model_engine(model_vit, model_name="ViT", epochs=15)

## 4. Train EfficientNet-V2-S with Huber loss

In [ ]:
import torch.nn.functional as F

torch.cuda.empty_cache()
gc.collect()

print(" 正在构建 EfficientNet-V2 多任务回归模型 (Huber loss)...")
cnn_base = models.efficientnet_v2_s(weights='DEFAULT')
in_ftrs_cnn = cnn_base.classifier[1].in_features
cnn_base.classifier[1] = nn.Identity()

model_cnn = GZ2DecisionTreeModel(cnn_base, in_ftrs_cnn).to(device)

optimizer_cnn = torch.optim.AdamW(model_cnn.parameters(), lr=5e-5, weight_decay=0.01) # 降低初始学习率
scheduler_cnn = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_cnn, T_max=15, eta_min=1e-7)
scaler_cnn = torch.amp.GradScaler('cuda')
criterion_robust = nn.HuberLoss(delta=0.1) # 核心防爆盾

print(f"\n 开始训练 CNN | 目标: 15 轮")
best_cnn_loss = float('inf')

for epoch in range(15):
    start_time = time.time()
    model_cnn.train()
    running_loss = 0.0
    optimizer_cnn.zero_grad()
    
    pbar = tqdm(safe_train_loader, desc=f"CNN Epoch {epoch+1}/15", leave=True)
    for i, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model_cnn(images)
            loss = criterion_robust(outputs, labels) / 4
        
        scaler_cnn.scale(loss).backward()
        
        if (i + 1) % 4 == 0 or (i + 1) == len(safe_train_loader):
            scaler_cnn.unscale_(optimizer_cnn)
            torch.nn.utils.clip_grad_norm_(model_cnn.parameters(), max_norm=1.0) # 强力裁剪
            scaler_cnn.step(optimizer_cnn)
            scaler_cnn.update()
            optimizer_cnn.zero_grad()
            
        real_loss = loss.item() * 4
        running_loss += real_loss * images.size(0)
        pbar.set_postfix({'Loss': f"{real_loss:.4f}"})
        
    scheduler_cnn.step()
    
    model_cnn.eval()
    val_mse = 0.0
    with torch.no_grad():
        vbar = tqdm(val_loader, desc="Validating", leave=False)
        for images, labels in vbar:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model_cnn(images)
                mse_loss = F.mse_loss(outputs, labels)
            if not torch.isnan(mse_loss):
                val_mse += mse_loss.item() * images.size(0)
            
    epoch_val = val_mse / len(val_loader.dataset)
    print(f" CNN Epoch {epoch+1} | Val MSE: {epoch_val:.6f} | 耗时: {time.time()-start_time:.1f}s")
    
    if epoch_val < best_cnn_loss and epoch_val > 0:
        best_cnn_loss = epoch_val
        torch.save(model_cnn.state_dict(), str(CHECKPOINT_DIR / 'galaxy_CNN_tree.pth'))
        print(f"    [CNN] 模型权重已更新！RMSE: {np.sqrt(best_cnn_loss):.5f}")

## 5. Train Swin-T

In [ ]:
torch.cuda.empty_cache()
gc.collect()

print(" 正在构建 Swin Transformer 多任务回归模型...")
swin_base = models.swin_t(weights='DEFAULT')
in_ftrs_swin = swin_base.head.in_features
swin_base.head = nn.Identity()

model_swin = GZ2DecisionTreeModel(swin_base, in_ftrs_swin).to(device)

train_model_engine(model_swin, model_name="Swin", epochs=15)

## 6. Weighted ensemble with five-view TTA

The three models are evaluated on the same held-out local validation split. This is not an official Kaggle test evaluation.

In [ ]:
import torch.nn.functional as F

print(" 清理显存池，为三模型集成腾出空间...")
torch.cuda.empty_cache()
gc.collect()

vit_base = models.vit_b_16(); vit_base.heads.head = nn.Identity()
model_vit = GZ2DecisionTreeModel(vit_base, 768).to(device)
model_vit.load_state_dict(torch.load(str(CHECKPOINT_DIR / 'galaxy_ViT_tree.pth')))
model_vit.eval()

cnn_base = models.efficientnet_v2_s(); cnn_base.classifier[1] = nn.Identity()
model_cnn = GZ2DecisionTreeModel(cnn_base, 1280).to(device)
model_cnn.load_state_dict(torch.load(str(CHECKPOINT_DIR / 'galaxy_CNN_tree.pth')))
model_cnn.eval()

swin_base = models.swin_t(); swin_base.head = nn.Identity()
model_swin = GZ2DecisionTreeModel(swin_base, 768).to(device)
model_swin.load_state_dict(torch.load(str(CHECKPOINT_DIR / 'galaxy_Swin_tree.pth')))
model_swin.eval()

W_VIT, W_SWIN, W_CNN = 0.55, 0.30, 0.15

print(f" 执行三模型五视角TTA验证 (3 Models x 5 TTA)...")

running_val_loss = 0.0
total_samples = 0
criterion = nn.MSELoss()

def get_tta_prediction(net, imgs):
    p1 = net(imgs)
    p2 = net(torch.flip(imgs, dims=[3]))
    p3 = net(torch.flip(imgs, dims=[2]))
    p4 = net(torch.rot90(imgs, k=1, dims=[2, 3]))
    p5 = net(torch.rot90(imgs, k=2, dims=[2, 3]))
    return (p1 + p2 + p3 + p4 + p5) / 5.0

with torch.no_grad():
    vbar = tqdm(val_loader, desc="[Trinity Weighted Judgment]", leave=True)
    for images, labels in vbar:
        images, labels = images.to(device), labels.to(device)
        total_samples += images.size(0)
        
        with torch.amp.autocast('cuda'):
            vit_pred = get_tta_prediction(model_vit, images)
            swin_pred = get_tta_prediction(model_swin, images)
            cnn_pred = get_tta_prediction(model_cnn, images)
            
            final_pred = (W_VIT * vit_pred) + (W_SWIN * swin_pred) + (W_CNN * cnn_pred)
            loss = criterion(final_pred, labels)
            
        running_val_loss += loss.item() * images.size(0)

final_mse = running_val_loss / total_samples
final_rmse = np.sqrt(final_mse)

print("\n" + "="*60)
print(" 多任务回归模型三模型集成验证结果")
print("="*60)
print(f" 验证集 RMSE: {final_rmse:.5f}")

## 7. Collect validation predictions

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from tqdm.std import tqdm
from math import pi

print(" 启动全维扫描探针... 正在收集验证集最终推断数据...")

model_vit.eval(); model_swin.eval(); model_cnn.eval()

all_labels, all_preds_ensemble = [], []
all_preds_vit, all_preds_swin, all_preds_cnn = [], [], []

def get_tta(net, imgs):
    return (net(imgs) + net(torch.flip(imgs, dims=[3])) + 
            net(torch.flip(imgs, dims=[2])) + 
            net(torch.rot90(imgs, k=1, dims=[2, 3])) + 
            net(torch.rot90(imgs, k=2, dims=[2, 3]))) / 5.0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="[Extracting TTA Outputs]"):
        images, labels = images.to(device), labels.to(device)
        
        with torch.amp.autocast('cuda'):
            p_vit = get_tta(model_vit, images)
            p_swin = get_tta(model_swin, images)
            p_cnn = get_tta(model_cnn, images)
            p_ens = 0.55 * p_vit + 0.30 * p_swin + 0.15 * p_cnn
            
        all_labels.append(labels.cpu().numpy())
        all_preds_ensemble.append(p_ens.cpu().numpy())
        all_preds_vit.append(p_vit.cpu().numpy())
        all_preds_swin.append(p_swin.cpu().numpy())
        all_preds_cnn.append(p_cnn.cpu().numpy())

Y_true = np.vstack(all_labels)
Y_pred = np.vstack(all_preds_ensemble)
P_vit = np.vstack(all_preds_vit)
P_swin = np.vstack(all_preds_swin)
P_cnn = np.vstack(all_preds_cnn)

print(" 区块一完成：三个模型的TTA预测矩阵已收集！")

## 8. Aggregate output diagnostics

These summary statistics are descriptive. Matching their means does not establish per-image probability conservation or conditional consistency.

In [ ]:
t1_sum = np.sum(Y_pred[:, 0:3], axis=1)
t2_sum = np.sum(Y_pred[:, 3:5], axis=1)
parent_t1_disk = Y_pred[:, 1]
dependency_ratio = np.mean(t2_sum / (parent_t1_disk + 1e-8))
print(f'Mean root-head output sum: {np.mean(t1_sum):.4f}')
print(f'Mean task-2/root-disk output ratio: {dependency_ratio:.4f}')

## 9. Validation metrics

In [ ]:
print("\n" + "="*60)
print("  验证集指标 (Six-Dimensional Metrics)")
print("="*60)

global_rmse = np.sqrt(np.mean((Y_pred - Y_true)**2))
print(f"1. 全局均方根误差 (Global RMSE): {global_rmse:.5f} ")

r2 = r2_score(Y_true, Y_pred)
print(f"2. 决定系数 (R² Score): {r2:.4f} (解释了 {r2*100:.1f}% 的志愿者投票方差)")

pearson_corr = np.mean([pearsonr(Y_pred[i], Y_true[i])[0] for i in range(len(Y_true))])
print(f"3. 皮尔逊相关系数 (Pearson Correlation): {pearson_corr:.4f}")

dot_product = np.sum(Y_pred * Y_true, axis=1)
norm_pred = np.linalg.norm(Y_pred, axis=1)
norm_true = np.linalg.norm(Y_true, axis=1)
cos_sim = np.mean(dot_product / (norm_pred * norm_true + 1e-8))
print(f"4. 37维空间余弦相似度 (Cosine Similarity): {cos_sim:.4f}")

root_acc = np.mean(np.argmax(Y_pred[:, 0:3], axis=1) == np.argmax(Y_true[:, 0:3], axis=1))
print(f"5. 根节点分类准确率 (Root Node Accuracy): {root_acc * 100:.2f}%")

task_slices = [(0,3), (3,5), (5,7), (7,9), (9,13), (13,15), (15,18), (18,25), (25,28), (28,31), (31,37)]
task_names = ['T1:Smooth', 'T2:EdgeOn', 'T3:Bar', 'T4:Spiral', 'T5:Bulge', 'T6:Odd', 'T7:Round', 'T8:OddType', 'T9:BulgeShape', 'T10:ArmWind', 'T11:ArmNum']
task_rmses_ens = [np.sqrt(np.mean((Y_pred[:, s:e] - Y_true[:, s:e])**2)) for s, e in task_slices]

print("6. 节点独立 RMSE 剖析:")
for name, rmse in zip(task_names, task_rmses_ens):
    print(f"   - {name}: {rmse:.5f}")

## 10. Task-specific error comparison

All four series are calculated from the collected validation predictions. No synthetic ensemble convergence curve or cross-dataset leaderboard comparison is used.

In [ ]:
rmses_vit = [np.sqrt(np.mean((P_vit[:, s:e] - Y_true[:, s:e]) ** 2)) for s, e in task_slices]
rmses_swin = [np.sqrt(np.mean((P_swin[:, s:e] - Y_true[:, s:e]) ** 2)) for s, e in task_slices]
rmses_cnn = [np.sqrt(np.mean((P_cnn[:, s:e] - Y_true[:, s:e]) ** 2)) for s, e in task_slices]
for name, value in zip(task_names, task_rmses_ens):
    print(f'{name}: {value:.5f}')

fig, ax = plt.subplots(figsize=(12, 5))
for label, values in [('ViT', rmses_vit), ('Swin', rmses_swin), ('CNN', rmses_cnn), ('Ensemble', task_rmses_ens)]:
    ax.plot(range(len(task_names)), values, marker='o', label=label)
ax.set_xticks(range(len(task_names)), task_names, rotation=40, ha='right')
ax.set_ylabel('Validation RMSE')
ax.set_title('Task-specific validation error')
ax.legend()
fig.tight_layout()
plt.show()

## 11. RMSE and MAE

In [ ]:
global_mae = np.mean(np.abs(Y_true - Y_pred))
global_rmse = np.sqrt(np.mean((Y_true - Y_pred)**2))

print("="*60)
print(f" Method 1: Global Error Magnitude")
print("="*60)
print(f"   - Root Mean Square Error (RMSE): {global_rmse:.5f}")
print(f"   - Mean Absolute Error (MAE): {global_mae:.5f}")
print(f"   - RMSE/MAE Ratio: {global_rmse/global_mae:.3f}")

plt.figure(figsize=(7, 5), dpi=100)
plt.bar(['RMSE (L2)', 'MAE (L1)'], [global_rmse, global_mae], color=['#2980b9', '#3498db'], alpha=0.8, width=0.6)
plt.title("Method 1: Global Error Metric Comparison", fontweight='bold')
plt.ylabel("Error Score")
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Explained variance and regression scatter

In [ ]:
from sklearn.metrics import explained_variance_score
exp_var = explained_variance_score(Y_true, Y_pred)

print("="*60)
print(f" Method 2: Variance & Fit Consistency")
print("="*60)
print(f"   - Explained Variance Score: {exp_var:.4f}")
print(f"   - Unexplained Variance: {1 - exp_var:.4f}")

plt.figure(figsize=(7, 7), dpi=100)
plt.scatter(Y_true[:1000, 0], Y_pred[:1000, 0], alpha=0.2, color='#e67e22', s=12, label='Samples')
plt.plot([0, 1], [0, 1], color='#2c3e50', linestyle='--', linewidth=2, label='Perfect Fit (y=x)')
plt.title("Method 2: Regression Consistency (Task 1: Smooth)", fontweight='bold')
plt.xlabel("Ground Truth (Volunteer Vote Proportion)")
plt.ylabel("Model Predicted Probability")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 13. Sample-wise error distribution

In [ ]:
sample_mse = np.mean((Y_true - Y_pred)**2, axis=1)
q = [50, 90, 95, 99]
percentiles = np.percentile(sample_mse, q)

print("="*60)
print(f" Method 3: Sample-wise Error Quantiles")
print("="*60)
for p, val in zip(q, percentiles):
    print(f"   - {p}th percentile (MSE): {val:.6f}")

plt.figure(figsize=(9, 5), dpi=100)
sorted_mse = np.sort(sample_mse)
cdf = np.arange(1, len(sorted_mse) + 1) / float(len(sorted_mse))

plt.plot(sorted_mse, cdf, color='#c0392b', linewidth=2.5, label='Error CDF')
plt.axvline(x=percentiles[2], color='#7f8c8d', linestyle='--', label='95% Threshold')
plt.xscale('log') # 使用对数坐标以便观察低误差区的细节
plt.title("Method 3: Cumulative Error Distribution (CED)", fontweight='bold')
plt.xlabel("Sample MSE (Log Scale)")
plt.ylabel("Cumulative Proportion of Samples")
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.tight_layout()
plt.show()

## 14. Mean residuals

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

all_37_task_names = [
    "T1_Smooth", "T1_Features", "T1_Artifact", 
    "T2_EdgeOn_Yes", "T2_EdgeOn_No", 
    "T3_Bar_Yes", "T3_Bar_No", 
    "T4_Spiral_Yes", "T4_Spiral_No", 
    "T5_Bulge_None", "T5_Bulge_Small", "T5_Bulge_Mod", "T5_Bulge_Large",
    "T6_Odd_Yes", "T6_Odd_No", 
    "T7_Round", "T7_InBetween", "T7_Cigar",
    "T8_Ring", "T8_Lens", "T8_Disturbed", "T8_Irregular", "T8_Other", "T8_Merger", "T8_DustLane",
    "T9_Bulge_Rounded", "T9_Bulge_Boxy", "T9_Bulge_None",
    "T10_Arms_Tight", "T10_Arms_Med", "T10_Arms_Loose",
    "T11_Arms_1", "T11_Arms_2", "T11_Arms_3", "T11_Arms_4", "T11_Arms_More", "T11_Arms_Unknown"
]

task_bias = np.mean(Y_true - Y_pred, axis=0) 

max_abs_bias = np.max(np.abs(task_bias))
v_limit = max(max_abs_bias, 0.005) 

print("="*60)
print(f" Method 4: High-Sensitivity Systematic Bias")
print("="*60)
print(f"   - Current Auto-Scale Range: ±{v_limit:.5f}")

plt.figure(figsize=(16, 5), dpi=300)
im = plt.imshow(task_bias.reshape(1, -1), cmap='RdBu_r', aspect='auto', vmin=-v_limit, vmax=v_limit)

cbar = plt.colorbar(im, orientation='horizontal', pad=0.45)
cbar.set_label(f'Mean Residual Error (True - Pred) | Scale: ±{v_limit:.4f}', fontweight='bold')

plt.xticks(np.arange(37), all_37_task_names, rotation=90, fontsize=9)
plt.yticks([]) 

for i, val in enumerate(task_bias):
    if abs(val) > v_limit * 0.7: # 只高亮显示比较明显的偏差
        plt.text(i, 0, f'{val*1000:.1f}m', ha='center', va='center', 
                 color='white' if abs(val) > v_limit * 0.8 else 'black', 
                 fontsize=7, fontweight='bold')

plt.title("Method 4: Systematic Bias Heatmap (Auto-Scaled Perspective)", fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig("Report_Fig_Method4_HighSens_Bias.png", bbox_inches='tight')
plt.show()

print(f"\n 解读提示：")
print(f"   - 数值为正 (红色): 模型在该特征上预测过于‘保守’，预测值低于人类平均水平。")
print(f"   - 数值为负 (蓝色): 模型在该特征上预测过于‘激进’，预测值高于人类平均水平。")

## 15. Binned calibration diagnostic

In [ ]:
bins = np.linspace(0, 1, 11)
bin_centers = (bins[:-1] + bins[1:]) / 2
true_freqs = []

print("="*60)
print(f" Method 5: Calibration Table (Task 1: Smooth)")
print("="*60)
for i in range(len(bins)-1):
    mask = (Y_pred[:, 0] >= bins[i]) & (Y_pred[:, 0] < bins[i+1])
    count = np.sum(mask)
    if count > 0:
        freq = np.mean(Y_true[mask, 0])
        true_freqs.append(freq)
        print(f"   - Bin [{bins[i]:.1f}-{bins[i+1]:.1f}]: Samples={count:<4} | Observed Freq={freq:.4f}")
    else:
        true_freqs.append(np.nan)

plt.figure(figsize=(7, 7), dpi=100)
plt.plot([0, 1], [0, 1], color='#bdc3c7', linestyle=':', linewidth=2, label='Perfectly Calibrated')
plt.plot(bin_centers, true_freqs, marker='s', color='#16a085', markersize=8, linewidth=2, label='Ensemble Model')
plt.title("Method 5: Probability Calibration (Reliability Diagram)", fontweight='bold')
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Human Vote Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Highest-error examples

In [ ]:
worst_indices = np.argsort(sample_mse)[-5:][::-1]

print("="*60)
print(f" Method 6: Hall of Outliers (Top 5 Failure Cases)")
print("="*60)
for i, idx in enumerate(worst_indices):
    img_id = str(int(val_df.iloc[idx, 0]))
    print(f"   - Rank {i+1} Worst Case: ID={img_id:<10} | Sample MSE={sample_mse[idx]:.6f}")

plt.figure(figsize=(16, 5), dpi=100)
for i, idx in enumerate(worst_indices):
    img_id = str(int(val_df.iloc[idx, 0]))
    img_path = os.path.join(IMG_DIR, f"{img_id}.jpg")
    
    ax = plt.subplot(1, 5, i+1)
    try:
        img = Image.open(img_path)
        plt.imshow(img)
    except FileNotFoundError:
        plt.text(0.5, 0.5, f"Image {img_id}\nNot Found", ha='center', va='center')
    
    plt.title(f"Rank {i+1} Worst\nMSE: {sample_mse[idx]:.4f}", color='#c0392b', fontweight='bold')
    plt.axis('off')

plt.suptitle("Method 6: Visual Inspection of Highest-Error Outliers", fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()